In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import numpy as np
from numpy.polynomial import legendre as npleg  # only used to check mlcore

import mlcore as mc

rng = np.random.default_rng(407)

## Unit tests for `mlcore`

The cell below checks every helper in `mlcore.py` with small `assert`s:

- **`add_bias`**: output shape for 1D and 2D input, first column all ones.
- **`legendre_features`**: matches `numpy.polynomial.legendre.legval` for Q = 0..10 on 200 random x in [-1, 1] (max abs error < 1e-10).
- **`poly_features`**: `X = [[2, 3]]`, Q = 2 gives exactly {2, 3, 4, 6, 9}; 2 inputs with Q = 3 gives 9 columns.
- **`standardize_fit` / `standardize_apply`**: training output has mean ≈ 0 and std ≈ 1; a constant column gives 0, not `nan`.
- **`kfold_indices`**: each index is in exactly one validation fold, train and val are disjoint, fold sizes differ by at most 1.
- **Metrics** on the hand example y = [1, 1, -1, -1, 1], ŷ = [1, -1, -1, 1, 1]: TP = 2, FP = 1, FN = 1, TN = 1, precision = recall = F1 = 2/3, accuracy = 0.6.
- **`roc_curve` / `auc`**: AUC = 1 for perfectly ranked scores, 0 for reversed, ≈ 0.5 for random scores on 10 000 points; curve runs from (0, 0) to (1, 1) and tied scores are grouped.
- **`sigmoid` / `logistic_Ein`**: `sigmoid(±1000)` is exactly 1 / 0 with no warnings or `nan`; `logistic_Ein` stays finite for huge w.
- **`gradient_check`**: relative error < 1e-6 for `linreg_grad` and `logistic_grad` on random data (N = 50, d = 5, with bias).

In [ ]:
# --- add_bias ---
X = rng.normal(size=(7, 3))
Xb = mc.add_bias(X)
assert Xb.shape == (7, 4)
assert np.all(Xb[:, 0] == 1) and np.array_equal(Xb[:, 1:], X)
assert mc.add_bias(np.arange(5.0)).shape == (5, 2)

# --- legendre_features vs numpy.polynomial.legendre.legval ---
x = rng.uniform(-1, 1, size=200)
for Q in range(11):
    L = mc.legendre_features(x, Q)
    assert L.shape == (200, Q + 1)
    for k in range(Q + 1):
        c = np.zeros(k + 1); c[k] = 1
        assert np.max(np.abs(L[:, k] - npleg.legval(x, c))) < 1e-10, (Q, k)

# --- poly_features ---
Phi, exps = mc.poly_features(np.array([[2.0, 3.0]]), 2, return_exponents=True)
assert Phi.shape == (1, 5)
assert sorted(Phi[0].tolist()) == [2, 3, 4, 6, 9]
assert exps == [(1, 0), (0, 1), (2, 0), (1, 1), (0, 2)]
assert np.array_equal(Phi[0], [2, 3, 4, 6, 9])            # documented order
assert mc.poly_features(rng.normal(size=(4, 2)), 3).shape == (4, 9)

# --- standardize ---
Xtr = rng.normal(5, 3, size=(100, 4))
Xtr[:, 2] = 7.0                                           # constant column
mu, sd = mc.standardize_fit(Xtr)
Z = mc.standardize_apply(Xtr, mu, sd)
assert not np.any(np.isnan(Z))
assert np.allclose(Z.mean(axis=0), 0, atol=1e-12)
assert np.allclose(np.delete(Z.std(axis=0), 2), 1)
assert np.all(Z[:, 2] == 0)

# --- kfold_indices ---
N, K = 103, 5
folds = mc.kfold_indices(N, K, rng)
assert len(folds) == K
all_val = np.concatenate([v for _, v in folds])
assert np.array_equal(np.sort(all_val), np.arange(N))     # each index once
for tr, va in folds:
    assert len(np.intersect1d(tr, va)) == 0
    assert len(tr) + len(va) == N
sizes = [len(v) for _, v in folds]
assert max(sizes) - min(sizes) <= 1

# --- classification metrics, hand example ---
y    = np.array([1, 1, -1, -1, 1])
yhat = np.array([1, -1, -1, 1, 1])
assert mc.confusion_matrix(y, yhat) == {"TP": 2, "FP": 1, "FN": 1, "TN": 1}
assert np.isclose(mc.precision(y, yhat), 2 / 3)
assert np.isclose(mc.recall(y, yhat), 2 / 3)
assert np.isclose(mc.f1(y, yhat), 2 / 3)
assert np.isclose(mc.accuracy(y, yhat), 0.6)
assert np.isclose(mc.error(y, yhat), 0.4)
# no positive predictions -> 0.0, not a division error
assert mc.precision(y, -np.ones(5)) == 0.0 and mc.f1(y, -np.ones(5)) == 0.0

# --- ROC / AUC ---
yr = np.array([1, 1, 1, -1, -1, -1])
sr = np.array([0.9, 0.8, 0.7, 0.3, 0.2, 0.1])
fpr, tpr, thr = mc.roc_curve(yr, sr)
assert (fpr[0], tpr[0]) == (0, 0) and (fpr[-1], tpr[-1]) == (1, 1)
assert np.isinf(thr[0])
assert mc.auc(fpr, tpr) == 1.0
assert mc.auc(*mc.roc_curve(yr, -sr)[:2]) == 0.0
# all scores tied -> one diagonal step, AUC = 0.5 regardless of order
fpr_t, tpr_t, _ = mc.roc_curve(yr, np.zeros(6))
assert np.array_equal(fpr_t, [0, 1]) and np.array_equal(tpr_t, [0, 1])
yrand = rng.choice([-1, 1], size=10000)
auc_rand = mc.auc(*mc.roc_curve(yrand, rng.normal(size=10000))[:2])
assert abs(auc_rand - 0.5) < 0.03, auc_rand

# --- sigmoid and logistic_Ein stability ---
with warnings.catch_warnings(), np.errstate(over="raise", invalid="raise", divide="raise"):  # underflow to 0 is harmless
    warnings.simplefilter("error")
    assert mc.sigmoid(1000) == 1.0 and mc.sigmoid(-1000) == 0.0
    s = mc.sigmoid(np.array([-1000.0, 0.0, 1000.0]))
    assert not np.any(np.isnan(s)) and s[1] == 0.5
    Xh = mc.add_bias(rng.normal(size=(20, 3)))
    yh = rng.choice([-1, 1], size=20)
    assert np.isfinite(mc.logistic_Ein(np.full(4, 1e6), Xh, yh))
    assert np.all(np.isfinite(mc.logistic_grad(np.full(4, 1e6), Xh, yh)))

# --- gradient checks (N = 50, d = 5, with bias) ---
Xg = mc.add_bias(rng.normal(size=(50, 5)))
wg = rng.normal(size=6)
y_reg = rng.normal(size=50)
y_cls = rng.choice([-1, 1], size=50)
err_lin = mc.gradient_check(lambda w: mc.linreg_Ein(w, Xg, y_reg),
                            lambda w: mc.linreg_grad(w, Xg, y_reg), wg)
err_log = mc.gradient_check(lambda w: mc.logistic_Ein(w, Xg, y_cls),
                            lambda w: mc.logistic_grad(w, Xg, y_cls), wg)
assert err_lin < 1e-6, err_lin
assert err_log < 1e-6, err_log
print(f"gradient_check: linreg {err_lin:.2e}, logistic {err_log:.2e}")
print(f"AUC on random scores: {auc_rand:.4f}")

print("All tests passed")